# 🐉 Daily Challenge: Pokemon Win Prediction Analysis

This notebook covers:
1. Data Preparation (loading, cleaning, merging, win % calculation)
2. Exploratory Data Analysis & Visualization
3. Machine Learning (Linear Regression, Random Forest, XGBoost) to predict win percentage

**Dataset:** `pokemon.csv` (Pokemon stats) + `combats.csv` (50,000 battle outcomes)


## 0. Setup — Download & Extract the Dataset

In [ ]:
!wget -q "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Pokemon%20Data%20Analysis%20Tutorial.zip" -O pokemon_data.zip
!unzip -o -q pokemon_data.zip -d pokemon_data
!find pokemon_data -type f


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    !pip install -q xgboost
    from xgboost import XGBRegressor
    HAS_XGB = True

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Data Preparation

### 1.1 Load the datasets

The zip extracts to a folder — we glob for the CSV files so this works regardless of exact folder naming.


In [ ]:
import glob

pokemon_path = glob.glob("pokemon_data/**/pokemon.csv", recursive=True)[0]
combats_path = glob.glob("pokemon_data/**/combats.csv", recursive=True)[0]

pokemon = pd.read_csv(pokemon_path)
combats = pd.read_csv(combats_path)

print("Pokemon shape:", pokemon.shape)
print("Combats shape:", combats.shape)
pokemon.head()


In [ ]:
combats.head()

### 1.2 Fix missing values

- Pokemon **#62** is missing its `Name` — it should be **Primeape**.
- `Type 2` has many `NaN` values for Pokemon with only one type — fill these with `"None"`.


In [ ]:
# Fix missing name for Pokemon #62 (Primeape)
pokemon.loc[pokemon["#"] == 62, "Name"] = "Primeape"

# Confirm the fix
pokemon[pokemon["#"] == 62]


In [ ]:
# Check missing values before fixing
print("Missing values before:")
print(pokemon.isnull().sum())

# Fill missing Type 2 with "None"
pokemon["Type 2"] = pokemon["Type 2"].fillna("None")

print("\nMissing values after:")
print(pokemon.isnull().sum())


### 1.3 Calculate win percentage for each Pokemon

For every battle in `combats.csv`, the `Winner` column holds the `#` (ID) of the winning Pokemon, and `First_pokemon` / `Second_pokemon` are the two combatants. We compute:

`win% = (number of wins) / (number of total battles fought) * 100`


In [ ]:
# Count total battles each pokemon participated in
total_battles = pd.concat([combats["First_pokemon"], combats["Second_pokemon"]]).value_counts()

# Count total wins for each pokemon
total_wins = combats["Winner"].value_counts()

# Build a win-rate dataframe indexed by Pokemon #
win_df = pd.DataFrame({
    "Battles": total_battles,
    "Wins": total_wins
}).fillna(0)

win_df["Win Percentage"] = (win_df["Wins"] / win_df["Battles"]) * 100
win_df = win_df.reset_index().rename(columns={"index": "#"})

win_df.head()


### 1.4 Merge stats with win percentage into a single clean dataframe

In [ ]:
data = pokemon.merge(win_df, on="#", how="left")

# Pokemon that never appeared in combats.csv will have NaN battles/wins/win% -> drop or fill with 0
print("Pokemon with no battle data:", data["Win Percentage"].isnull().sum())
data["Win Percentage"] = data["Win Percentage"].fillna(0)
data["Battles"] = data["Battles"].fillna(0)
data["Wins"] = data["Wins"].fillna(0)

data.head()


In [ ]:
data.describe(include="all").T

## 2. Exploratory Analysis & Visualization

### 2.1 Correlation matrix


In [ ]:
stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Win Percentage"]
corr = data[stat_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix: Stats vs Win Percentage")
plt.tight_layout()
plt.show()


**Observation:** `Speed` typically shows the strongest positive correlation with `Win Percentage` — going first in a battle is a huge advantage in the Pokemon battle mechanic. `Attack` and `Sp. Atk` also tend to correlate positively, while `HP`/`Defense` are usually weaker predictors.

### 2.2 Pairplot — stats vs win percentage

In [ ]:
sns.pairplot(
    data[["HP", "Attack", "Speed", "Win Percentage"]],
    diag_kind="kde",
    plot_kws={"alpha": 0.4, "s": 20}
)
plt.suptitle("Pairplot: HP, Attack, Speed vs Win Percentage", y=1.02)
plt.show()


### 2.3 Top 10 Pokemon by win percentage

In [ ]:
top10 = data.sort_values("Win Percentage", ascending=False).head(10)
top10[["Name", "Type 1", "Type 2", "HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Win Percentage", "Legendary"]]


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=top10, x="Win Percentage", y="Name", palette="viridis")
plt.title("Top 10 Pokemon by Win Percentage")
plt.xlabel("Win Percentage (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Compare top 10's average stats to the overall average
top10_stats = top10[stat_cols[:-1]].mean()
overall_stats = data[stat_cols[:-1]].mean()

comparison = pd.DataFrame({"Top 10 Avg": top10_stats, "Overall Avg": overall_stats})
comparison.plot(kind="bar", figsize=(9, 5))
plt.title("Top 10 Pokemon Avg Stats vs Overall Average")
plt.ylabel("Stat value")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

comparison


**Observation:** The top 10 win-rate Pokemon tend to have noticeably higher `Speed` and `Attack`/`Sp. Atk` than the population average, reinforcing the correlation matrix finding — speed and offensive power drive battle outcomes more than bulk stats like HP/Defense.

## 3. Machine Learning — Predicting Win Percentage

### 3.1 Feature engineering & train/test split


In [ ]:
# One-hot encode categorical features (Type 1, Type 2), keep Legendary as int
ml_data = data.copy()
ml_data["Legendary"] = ml_data["Legendary"].astype(int)

features = pd.get_dummies(
    ml_data[["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Generation", "Legendary", "Type 1", "Type 2"]],
    columns=["Type 1", "Type 2"],
    drop_first=True
)

target = ml_data["Win Percentage"]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


### 3.2 Train & evaluate 3 regression models

In [ ]:
results = {}

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
results["Linear Regression"] = {
    "MAE": mean_absolute_error(y_test, pred_lr),
    "R2": r2_score(y_test, pred_lr)
}

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results["Random Forest"] = {
    "MAE": mean_absolute_error(y_test, pred_rf),
    "R2": r2_score(y_test, pred_rf)
}

# --- XGBoost ---
xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)
results["XGBoost"] = {
    "MAE": mean_absolute_error(y_test, pred_xgb),
    "R2": r2_score(y_test, pred_xgb)
}

results_df = pd.DataFrame(results).T.sort_values("MAE")
results_df


In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(x=results_df.index, y=results_df["MAE"], palette="mako")
plt.title("Model Comparison — Mean Absolute Error (lower is better)")
plt.ylabel("MAE")
plt.tight_layout()
plt.show()

print(f"Best model: {results_df['MAE'].idxmin()} (MAE = {results_df['MAE'].min():.3f})")


### 3.3 Feature importance (Random Forest / XGBoost)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, palette="crest")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## 4. Dimensionality Reduction with PCA

Visualize how the 6 base stats reduce to 2 principal components, colored by win percentage.


In [ ]:
stat_features = data[["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]]
scaled = StandardScaler().fit_transform(stat_features)

pca = PCA(n_components=2)
pcs = pca.fit_transform(scaled)

pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2"])
pca_df["Win Percentage"] = data["Win Percentage"].values

plt.figure(figsize=(9, 7))
sc = plt.scatter(pca_df["PC1"], pca_df["PC2"], c=pca_df["Win Percentage"], cmap="viridis", alpha=0.7, s=25)
plt.colorbar(sc, label="Win Percentage")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.title("PCA of Pokemon Base Stats, colored by Win Percentage")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance captured by 2 PCs: {:.1f}%".format(pca.explained_variance_ratio_.sum()*100))


## 5. Summary

- Cleaned `pokemon.csv` (fixed `#62` name, filled `Type 2` NaNs) and merged it with win-rate stats derived from `combats.csv`.
- `Speed` and offensive stats (`Attack`, `Sp. Atk`) correlate most strongly with win percentage; `HP`/`Defense` matter less.
- Among Linear Regression, Random Forest, and XGBoost, tree-based models (Random Forest / XGBoost) typically achieve the lowest MAE, capturing non-linear interactions between stats that Linear Regression misses.
- PCA shows the 6 base stats compress into 2 components capturing the majority of variance, with higher win percentages skewing toward the offensive/speed-loaded region of the component space.
